# ⚗️ OpenMolcas on Google Colab

Run [OpenMolcas](https://gitlab.com/Molcas/OpenMolcas) quantum chemistry calculations directly in the browser — no installation required.

**Workflow — run each cell top to bottom using the ▶ button:**
0. **Connect to runtime** — click **Connect** (top right corner) before running any cell
1. **Setup** — downloads and configures OpenMolcas (~30 s, once per session)
2. **Configuration** — set a name for your calculation
3. **Input file** — write or edit your molecular input
4. **Run** — executes the calculation
5. **Output file** — full OpenMolcas output, scrollable
6. **Results** — parsed energies and properties
7. **Geometry** — 3-D molecular viewer
8. **Orbital viewer** — all MOs, isovalue slider, electron density
9. **Download** — zip of all output files

**Methods available:** `&SCF` · `&RASSCF`/`&CASSCF` · `&CASPT2` · `&NEVPT2` · `&MP2` · `&RASSI` · `&ALASKA`/`&SLAPAF`

---
*Part of the [OPAL suite](https://github.com/LucaDeVico/openmolcas-colab) — OpenMolcas tools for teaching and research*

> 💡 **To keep your changes:** use **File → Save a copy in Drive**.
> This notebook opens read-only from GitHub — your edits are not saved automatically.

In [ ]:
# @title 🔧  Setup — run once per session
import os, glob

if not os.path.isdir("/opt/openmolcas"):
    print("Downloading OpenMolcas binaries...")
    !wget -q --show-progress -O /tmp/omcas.tar.gz "https://github.com/LucaDeVico/openmolcas-colab/releases/download/build-20260611/openmolcas-colab-ubuntu22-20260611.tar.gz"
    print("Unpacking...")
    !tar -xzf /tmp/omcas.tar.gz -C /opt/
    !rm /tmp/omcas.tar.gz
    !apt-get install -y -qq libhdf5-dev
    !pip install -q py3Dmol ipywidgets
else:
    print("✓ OpenMolcas already present.")
    !pip install -q py3Dmol ipywidgets 2>/dev/null

if not os.path.isfile("/opt/openmolcas/molcas_viewer.html"):
    !wget -q -O /opt/openmolcas/molcas_viewer.html "https://raw.githubusercontent.com/LucaDeVico/openmolcas-colab/main/molcas_viewer.html"
    print("✓ Orbital viewer downloaded.")
else:
    print("✓ Orbital viewer present.")

hits = (glob.glob("/opt/openmolcas/bin/pymolcas") +
        glob.glob("/opt/openmolcas/pymolcas"))
if not hits:
    raise RuntimeError("pymolcas not found — check the release URL or re-run setup.")
PYMOLCAS = hits[0]

WORK_ROOT = "/content/SCRATCH"
os.makedirs(WORK_ROOT, exist_ok=True)
os.environ.update({
    "MOLCAS":           "/opt/openmolcas",
    "MOLCAS_DRIVER":    PYMOLCAS,
    "MOLCAS_MOLDEN":    "ON",
    "MOLCAS_NTHREADS":  "2",
    "OMP_NUM_THREADS":  "2",
    "MOLCAS_MEM":       "1500",
    "MOLCAS_WORKDIR":   WORK_ROOT,
    "MOLCAS_LINK":      "Yes",
    "PATH":             os.path.dirname(PYMOLCAS) + ":" + os.environ.get("PATH", ""),
})
print(f"Environment ready.  pymolcas → {PYMOLCAS}")


In [ ]:
# @title ⚙️  Configuration
# @markdown Set a name for this calculation (used for output file names and the scratch folder):
JOB_NAME = "job01"  # @param {type:"string"}

import re
_ALLOWED = re.compile(r"^[A-Za-z0-9+\-_]+$")
if not JOB_NAME:
    raise ValueError("Calculation name cannot be empty.")
if not _ALLOWED.match(JOB_NAME):
    _bad = {repr(c) for c in JOB_NAME if not re.match(r"[A-Za-z0-9+\-_]", c)}
    raise ValueError(
        f"Invalid character(s) in calculation name: {', '.join(sorted(_bad))}\n"
        "Allowed: letters (A–Z, a–z), digits (0–9), and the symbols  +  -  _\n"
        "No spaces, accented letters, dots, or other special characters."
    )
if len(JOB_NAME) > 64:
    raise ValueError(f"Calculation name too long ({len(JOB_NAME)} chars, max 64).")
print(f"✓  Calculation name: '{JOB_NAME}'")


## Input file

Use the editor below to write your OpenMolcas input.

- **Coordinates** are in Ångström, XYZ format: number of atoms → title line → `element  x  y  z`
- Change `Basis` to any basis set known to OpenMolcas (e.g. `ano-rcc-vdzp`, `6-31G*`)
- Replace `&SCF` with `&RASSCF`, `&CASPT2`, etc. to use correlated methods
- Set `Group = C2v` (or other Schoenflies symbol) to exploit molecular symmetry

In [ ]:
# @title ✏️  Input file
import ipywidgets as widgets
from IPython.display import display

DEFAULT_INPUT = """\
&GATEWAY
 Coord
 3
 Water molecule
 O   0.000000  0.000000  0.000000
 H   0.758602  0.000000  0.504284
 H  -0.758602  0.000000  0.504284
 Basis = cc-pVDZ
 Group = NoSym

&SEWARD

&SCF
 Charge = 0
 Spin = 1
"""

input_editor = widgets.Textarea(
    value=DEFAULT_INPUT,
    style={"description_width": "0px"},
    layout=widgets.Layout(width="100%", height="320px"),
)
display(widgets.HTML("<b>Input file</b> (edit freely, then run the next cell):"))
display(input_editor)

In [ ]:
# @title 📂  Upload orbital file — optional, for advanced use
# @markdown Upload a previously computed orbital file to use as starting orbitals.
# @markdown The file is renamed and placed in the scratch directory automatically.
# @markdown
# @markdown **Skip this cell entirely** for a fresh calculation from scratch.
# @markdown If you run it but upload nothing, the cell is safely ignored.
# @markdown
# @markdown Select the type of file you are uploading:
ORBITAL_TYPE = "RasOrb  (CASSCF / RASSCF)"  # @param ["RasOrb  (CASSCF / RASSCF)", "ScfOrb  (HF / DFT)", "UnaOrb  (UHF natural orbitals)", "Other — keep original filename"]

from google.colab import files
import os

# Maps dropdown choice → (file extension, keyword hint for the input)
_ORB_MAP = {
    "RasOrb  (CASSCF / RASSCF)": (
        "RasOrb",
        "Add  LUMORB  inside the &RASSCF block of your input."
    ),
    "ScfOrb  (HF / DFT)": (
        "ScfOrb",
        "Add  MOSTART = LUMORB  inside the &SCF block of your input."
    ),
    "UnaOrb  (UHF natural orbitals)": (
        "UnaOrb",
        "Add  LUMORB  inside the &RASSCF block of your input."
    ),
    "Other — keep original filename": (
        None,
        "Reference the file explicitly with  FILEORB = <filename>  in the relevant module."
    ),
}

print("Select your orbital file in the picker below:")
uploaded = files.upload()   # shows a file-picker widget; returns {filename: bytes}

if not uploaded:
    print("No file uploaded — skipping.")
else:
    work_dir = f"/content/SCRATCH/{JOB_NAME}"
    os.makedirs(work_dir, exist_ok=True)
    ext, hint = _ORB_MAP[ORBITAL_TYPE]

    for orig_name, data in uploaded.items():
        dest_name = f"{JOB_NAME}.{ext}" if ext else orig_name
        dest = os.path.join(work_dir, dest_name)
        with open(dest, "wb") as fh:
            fh.write(data)
        size_kb = len(data) / 1024
        print(f"✓  {orig_name}  →  {dest_name}  ({size_kb:.1f} kB)")

    print(f"\nTo use these orbitals in your input:")
    print(f"  {hint}")


In [ ]:
# @title ▶️  Run calculation
import os, subprocess, time

work_dir    = f"/content/SCRATCH/{JOB_NAME}"
input_file  = f"{work_dir}/{JOB_NAME}.input"
output_file = f"{work_dir}/{JOB_NAME}.output"
os.makedirs(work_dir, exist_ok=True)
os.environ["MOLCAS_WORKDIR"] = work_dir

with open(input_file, "w") as f:
    f.write(input_editor.value)

t0 = time.time()
print(f"Running {JOB_NAME} ...")
subprocess.run([PYMOLCAS, input_file], cwd=work_dir,
               stdout=open(output_file, "w"), stderr=subprocess.STDOUT)
elapsed = time.time() - t0

with open(output_file) as f:
    out_text = f.read()

ok = "Happy landing" in out_text
print(f"{'✓ Completed' if ok else '⚠  May have failed'}  ({elapsed:.1f} s)")
if not ok:
    print("\nLast 10 lines of output:")
    print("\n".join(out_text.split("\n")[-10:]))

In [ ]:
# @title 📄  Output file
import ipywidgets as widgets
from IPython.display import display

output_viewer = widgets.Textarea(
    value=open(output_file).read(),
    style={"description_width": "0px"},
    layout=widgets.Layout(width="100%", height="420px"),
)
output_viewer.disabled = True
display(widgets.HTML("<b>Output file</b> (read-only):"))
display(output_viewer)

In [ ]:
# @title 📊  Results
import re

def parse_output(path):
    res = {}
    with open(path) as f:
        text = f.read()
    lines = text.split("\n")
    res["completed"] = "Happy landing" in text
    for line in lines:
        if re.search(r"Total SCF energy", line):
            m = re.search(r"-\d+\.\d+", line)
            if m: res["scf_energy"] = float(m.group())
        if re.search(r"RASSCF root number.*Total energy", line):
            m = re.search(r"-\d+\.\d+", line)
            if m: res.setdefault("rasscf_energies", []).append(float(m.group()))
        if re.search(r"Total CASPT2 energy", line):
            m = re.search(r"-\d+\.\d+", line)
            if m: res.setdefault("caspt2_energies", []).append(float(m.group()))
        if re.search(r"Total dipole moment.*Debye", line):
            m = re.search(r"(\d+\.\d+)\s+Debye", line)
            if m: res["dipole_D"] = float(m.group(1))
    return res

results = parse_output(output_file)
sep = "─" * 52
print(sep)
print(f"Status:  {'✓ Normal termination' if results['completed'] else '⚠  Abnormal termination'}")
if "scf_energy"      in results:
    print(f"SCF energy:    {results['scf_energy']:.10f}  Eh")
if "rasscf_energies" in results:
    for i, e in enumerate(results["rasscf_energies"], 1):
        print(f"CASSCF root {i}: {e:.10f}  Eh")
if "caspt2_energies" in results:
    for i, e in enumerate(results["caspt2_energies"], 1):
        print(f"CASPT2 root {i}: {e:.10f}  Eh")
if "dipole_D"        in results:
    print(f"Dipole moment: {results['dipole_D']:.4f}  D")
print(sep)

In [ ]:
# @title 🔬  Geometry viewer
import py3Dmol

def input_to_xyz(text):
    lines, atoms = text.split("\n"), []
    in_coord, natoms, skip_title = False, 0, False
    i = 0
    while i < len(lines):
        if lines[i].strip().upper().startswith("COORD"):
            in_coord, skip_title = True, False
            i += 1
            while i < len(lines) and not lines[i].strip(): i += 1
            try: natoms = int(lines[i].strip())
            except (ValueError, IndexError): break
            i += 1; continue
        if in_coord and not skip_title:
            skip_title = True; i += 1; continue
        if in_coord and skip_title and natoms > 0:
            parts = lines[i].split()
            if len(parts) >= 4:
                try:
                    atoms.append((parts[0],float(parts[1]),float(parts[2]),float(parts[3])))
                    natoms -= 1
                except ValueError: pass
            if natoms == 0: break
        i += 1
    if not atoms: return None
    return f"{len(atoms)}\nOpenMolcas Colab\n" + "\n".join(
        f"{a}  {x:.6f}  {y:.6f}  {z:.6f}" for a,x,y,z in atoms)

xyz = input_to_xyz(input_editor.value)
if xyz:
    view = py3Dmol.view(width=520, height=420)
    view.addModel(xyz, "xyz")
    view.setStyle({"stick":{"radius":0.12},"sphere":{"radius":0.28}})
    view.setBackgroundColor("white")
    view.zoomTo(); view.show()
else:
    print("Could not parse geometry — check the GATEWAY / Coord block format.")

## Orbital viewer

Run the cell below to open the interactive orbital viewer directly in the notebook.
All molecular orbitals are available — including core orbitals — with isovalue slider,
colour schemes, electron density, and density difference between states.

**Controls:** rotate (left drag) · zoom (scroll) · pan (right drag) · click atom/bond to select

In [ ]:
# @title 🌐  Orbital viewer
import os, glob, json, threading, builtins
from http.server import HTTPServer, SimpleHTTPRequestHandler
from IPython.display import display, IFrame
from google.colab.output import eval_js

_SRV_PORT = 8765

def find_molden(directory):
    for pat in ['*.rasscf.molden','*.raspt2.molden','*.scf.molden','*.molden','*.molden2']:
        hits = sorted(glob.glob(os.path.join(directory, pat)))
        if hits: return hits[-1]
    return None

work_dir    = f"/content/SCRATCH/{JOB_NAME}"
molden_file = find_molden(work_dir)

if not molden_file:
    print(f"No Molden file found in {work_dir}")
    print(f"Files: {os.listdir(work_dir) if os.path.isdir(work_dir) else 'directory missing'}")
else:
    molden_text = open(molden_file).read()
    molden_name = os.path.basename(molden_file)
    print(f"Loading: {molden_name}  ({len(molden_text)//1024} kB)")

    template = open("/opt/openmolcas/molcas_viewer.html").read()
    viewer_html = (
        template
        .replace("const PRELOADED_MOLDEN   = null;",
                 f"const PRELOADED_MOLDEN   = {json.dumps(molden_text)};")
        .replace("const PRELOADED_FILENAME = null;",
                 f"const PRELOADED_FILENAME = {json.dumps(molden_name)};")
    )
    with open("/content/orbital_viewer.html","w") as f:
        f.write(viewer_html)

    if not getattr(builtins,"_orbital_server",None):
        class _H(SimpleHTTPRequestHandler):
            def __init__(self,*a,**kw): super().__init__(*a,directory="/content",**kw)
            def log_message(self,*a): pass
        builtins._orbital_server = HTTPServer(("localhost",_SRV_PORT),_H)
        threading.Thread(target=builtins._orbital_server.serve_forever,daemon=True).start()

    base_url   = eval_js(f"google.colab.kernel.proxyPort({_SRV_PORT})")
    viewer_url = f"{base_url}/orbital_viewer.html"
    display(IFrame(src=viewer_url, width="100%", height=680))

In [ ]:
# @title 📥  Download output files
from google.colab import files
import glob, zipfile, os

all_files = [f for f in glob.glob(f"/content/SCRATCH/{JOB_NAME}/**",recursive=True)
             if os.path.isfile(f)]

zip_path = f"/content/{JOB_NAME}_output.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for fp in all_files:
        zf.write(fp, os.path.relpath(fp,f"/content/SCRATCH/{JOB_NAME}"))

print(f"Zipped {len(all_files)} file(s) → {zip_path}\n")
for fp in sorted(all_files):
    print(f"  {os.path.basename(fp):40s}  {os.path.getsize(fp)/1024:8.1f} kB")
files.download(zip_path)